# Gene-level expression perturbation

Paper: Fig. 5e and Supplementary Fig. S7 (underlying optimized gene effects). Inputs: the original 1,133-donor feature DataFrame and final trained PeriClock model, supplied separately. Environment: Python/Scanpy, NumPy, pandas, SciPy/SLSQP, scikit-learn, joblib, Matplotlib and seaborn. Outputs: gene perturbation, cell-type details, individual results and full gene-results CSVs for four age groups. Validation: static only; the optimizations and the paper's assembled plots have not been rerun. This is the original analysis, not a generic replacement optimizer.

Original source indices below are **zero-based**. Original cell hashes are retained in cell metadata. Outputs were cleared; inclusion does not imply successful execution.

## Required resources and configuration

Shared `PeriClock-1000s_PeriClock_features_matrix.pkl`, final PeriClock model and licensed Arial fonts; [training matrices](../RESOURCE_DOWNLOAD.md#4-training-feature-matrices) and [models](../RESOURCE_DOWNLOAD.md#1-periclock-pretrained-models).

Download locations, sizes, SHA256, versions, permissions and publication status are maintained once in [RESOURCE_DOWNLOAD.md](../RESOURCE_DOWNLOAD.md). Individual GEO/GSM sources and sample choices are in the paper’s Supplementary Table 1. Clock models are uploaded to [Hugging Face Models](https://huggingface.co/LeyiZhang1/PeriClock); [Dataset uploads](https://huggingface.co/datasets/LeyiZhang1/PeriClock-data) are in progress. Both repositories are currently private. Consult the resource guide for per-file availability; donor-level public redistribution remains subject to review.

Set `PERICLOCK_RESOURCE_DIR` for external data and `PERICLOCK_MODEL_DIR` for models, or override individual file paths in the next cell. Keep outputs separate from inputs. Start a fresh kernel and run in order.


In [ ]:
from pathlib import Path
import os
# Start Jupyter from within this GitHub checkout.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "periclock").is_dir() and (p / "pyproject.toml").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook kernel from within the PeriClock code checkout")
RESOURCE_DIR = Path(os.environ.get("PERICLOCK_RESOURCE_DIR", "~/periclock-resources")).expanduser().resolve()
MODEL_DIR = Path(os.environ.get("PERICLOCK_MODEL_DIR", str(RESOURCE_DIR / "models"))).expanduser().resolve()
# Each individual file-path environment variable still overrides these defaults.
FEATURES_1000 = Path(os.environ.get("PERICLOCK_FEATURES_1000", str(RESOURCE_DIR / "datasets/PeriClock-1000s_PeriClock_features_matrix.pkl"))).expanduser().resolve()
MODEL_PATH = Path(os.environ.get("PERICLOCK_PERTURBATION_MODEL", str(MODEL_DIR / "PeriClock.pkl"))).expanduser().resolve()
FONT_DIR = Path(os.environ.get("PERICLOCK_FONT_DIR", "/path/to/Arial")).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get("PERICLOCK_TUTORIAL_OUTPUT", str(PROJECT_ROOT / "outputs/06_gene_level_perturbation"))).expanduser().resolve()
# Input files remain read-only; all relative output paths below use this directory.
for input_path in [FEATURES_1000, MODEL_PATH]:
    if not input_path.is_file():
        raise FileNotFoundError(f"Set the input path in this cell: {input_path}")
if not FONT_DIR.is_dir():
    raise FileNotFoundError("Set PERICLOCK_FONT_DIR to a directory containing licensed Arial fonts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(OUTPUT_DIR)

## Uniform fold change across cell types

Source: `final/fig5.ipynb`, cells 47, 50–53, 55, 60, 65 and 69, in source order. The original saved transforms, equal default gene weights, SLSQP objective and constraints, fold-change bounds 0.2–2.0, ten-year mean reduction, iteration/tolerance limits and core-gene thresholds are retained. Four original calls run the four predicted-age groups. Solver success and the full versus selected-core effects must be checked before accepting results. Outputs remain flat CSV files under `OUTPUT_DIR`, matching the source filenames. No prior tutorial variables or bootstrap results are required.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import joblib
import scanpy as sc
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

In [ ]:
import joblib
import numpy as np
import pandas as pd


# ============================================================
# 1. 加载原始特征表和训练好的pipeline
# ============================================================
df_all_features = joblib.load(
    str(FEATURES_1000)
)

pipeline = joblib.load(
    str(MODEL_PATH)
)


# ============================================================
# 2. 提取训练时保存的对象
# ============================================================
vt = pipeline["vt"]
skb = pipeline["skb"]

scaler_prop = pipeline["scaler_prop"]
scaler_expr = pipeline["scaler_expr"]

en_model = pipeline["model"]

prop_cols = [str(col) for col in pipeline["prop_cols"]]

expr_cols = [str(col) for col in pipeline["expr_cols"]]

selected_features = [str(col) for col in pipeline["selected_features"]]

# ============================================================
# 3. 基本检查
# ============================================================
required_cols = (
    prop_cols
    + expr_cols
    + ["Age"]
)

missing_cols = [
    col
    for col in required_cols
    if col not in df_all_features.columns
]

if missing_cols:
    raise ValueError(
        "df_all_features中缺少模型所需列：\n"
        + "\n".join(missing_cols[:30])
    )


# ============================================================
# 4. 构建目标变量
# ============================================================
y = pd.to_numeric(
    df_all_features["Age"],
    errors="coerce"
)

valid_mask = y.notna()

df_model = df_all_features.loc[
    valid_mask
].copy()

y = y.loc[
    valid_mask
].to_numpy(
    dtype=float
)


# ============================================================
# 5. 按训练时列顺序提取比例和表达特征
# ============================================================
X_prop_raw = df_model[
    prop_cols
].copy()

X_expr_raw = df_model[
    expr_cols
].copy()


# 确保全部为数值
X_prop_raw = X_prop_raw.apply(
    pd.to_numeric,
    errors="coerce"
)

X_expr_raw = X_expr_raw.apply(
    pd.to_numeric,
    errors="coerce"
)


# 检查缺失值
if X_prop_raw.isna().any().any():
    bad_cols = X_prop_raw.columns[
        X_prop_raw.isna().any()
    ].tolist()

    raise ValueError(
        "比例特征中存在无法转换为数值或缺失的数据：\n"
        + "\n".join(bad_cols[:20])
    )

if X_expr_raw.isna().any().any():
    bad_cols = X_expr_raw.columns[
        X_expr_raw.isna().any()
    ].tolist()

    raise ValueError(
        "表达特征中存在无法转换为数值或缺失的数据：\n"
        + "\n".join(bad_cols[:20])
    )


# ============================================================
# 6. 使用保存的vt进行变换
#
# 这里只能transform，不能重新fit
# ============================================================
X_expr_vt = vt.transform(
    X_expr_raw
)

vt_expr_cols = np.asarray(
    expr_cols,
    dtype=str
)[
    vt.get_support()
]

# ============================================================
# 7. 使用保存的skb进行变换
# ============================================================
X_expr_selected = skb.transform(
    X_expr_vt
)

selected_expr_cols = [
    str(col)
    for col in vt_expr_cols[
        skb.get_support()
    ]
]


# 检查列数与scaler是否一致
if X_expr_selected.shape[1] != len(
    scaler_expr.mean_
):
    raise ValueError(
        "表达特征数量与训练好的scaler_expr不一致："
        f"{X_expr_selected.shape[1]} vs "
        f"{len(scaler_expr.mean_)}"
    )

if X_prop_raw.shape[1] != len(
    scaler_prop.mean_
):
    raise ValueError(
        "比例特征数量与训练好的scaler_prop不一致："
        f"{X_prop_raw.shape[1]} vs "
        f"{len(scaler_prop.mean_)}"
    )


# ============================================================
# 8. 使用保存的scaler进行标准化
#
# 仍然只能transform
# ============================================================
X_prop_scaled = scaler_prop.transform(
    X_prop_raw
)

X_expr_scaled = scaler_expr.transform(
    X_expr_selected
)


# ============================================================
# 9. 合并为带列名的完整标准化矩阵
# ============================================================
all_cols = (
    prop_cols
    + selected_expr_cols
)

X_final = np.hstack([
    X_prop_scaled,
    X_expr_scaled
])

X_final_df = pd.DataFrame(
    X_final,
    columns=all_cols,
    index=df_model.index
)


# ============================================================
# 10. 提取ElasticNet最终使用的特征
# ============================================================
missing_selected = [
    feature
    for feature in selected_features
    if feature not in X_final_df.columns
]

if missing_selected:
    raise ValueError(
        "部分selected_features未出现在处理后的矩阵中：\n"
        + "\n".join(missing_selected[:30])
    )

X = X_final_df.loc[
    :,
    selected_features
].copy()

X.columns = X.columns.astype(str)

if hasattr(en_model, "feature_names_in_"):
    model_feature_names = [str(col) for col in en_model.feature_names_in_]

    missing_model_features = [feature for feature in model_feature_names if feature not in X.columns]

    if missing_model_features:
        raise ValueError(
            "X中缺少模型训练时的特征：\n"
            + "\n".join(
                missing_model_features[:30]
            )
        )

    X = X.loc[:, model_feature_names].copy()

    X.columns = X.columns.astype(str)
    
# ============================================================
# 11. 最终一致性检查
# ============================================================
if X.shape[1] != len(
    en_model.coef_
):
    raise ValueError(
        "X的特征数与ElasticNet系数数量不一致："
        f"{X.shape[1]} vs "
        f"{len(en_model.coef_)}"
    )

if list(X.columns) != selected_features:
    raise ValueError(
        "X的特征顺序与selected_features不一致。"
    )


# ============================================================
# 12. 验证预测
# ============================================================
pred_age = en_model.predict(X)

print("X shape:", X.shape)
print("Number of selected features:", len(selected_features))
print("Number of selected expression features:", len(selected_expr_cols))
print("Prediction range:", pred_age.min(), pred_age.max())

In [ ]:
import numpy as np
import pandas as pd

from scipy.optimize import minimize


def solve_uniform_fold_change_rejuvenation(
    X_target,
    en_model,
    scaler_expr,
    selected_expr_cols,
    feature_gene_map,
    reduce_years=10,
    candidate_genes=None,
    min_fold_change=0.2,
    max_fold_change=2.0,
    constraint_mode="mean",
    perturbation_tol=0.01,
    age_effect_tol=0.01,
    gene_weights=None,
    maxiter=500,
    ftol=1e-6,
    random_state=42
):
    """
    非细胞类型特异性的统一比例基因扰动。

    对每个gene只优化一个fold change，并把同一fold change
    应用于该gene在所有细胞类型中的表达特征。

    例如：
        AUTS2 fold_change = 0.6

    表示所有细胞类型中的AUTS2表达均降低40%。

    Parameters
    ----------
    X_target : pd.DataFrame
        目标年龄段样本的模型输入矩阵。
        必须是经过模型预处理后的最终标准化特征矩阵。

    en_model :
        已训练的ElasticNet模型。

    scaler_expr :
        表达特征使用的StandardScaler。

    selected_expr_cols : list
        scaler_expr对应的表达特征名称，顺序必须与
        scaler_expr.mean_和scale_一致。

    feature_gene_map : pd.Series or dict
        feature到gene的映射。

        例如：
            CD4Tem_S100A4_AUTS2 -> AUTS2
            CD8Tem_GZMK_AUTS2   -> AUTS2

    reduce_years : float
        目标预测年龄下降年数。

    candidate_genes : list or None
        允许扰动的候选基因。None表示所有可用基因。

    min_fold_change : float
        最低表达倍数。

        例如：
            0.2 = 最多敲低80%

    max_fold_change : float
        最高表达倍数。

        例如：
            2.0 = 最多过表达2倍

    constraint_mode : {"mean", "all"}
        mean：
            目标年龄段平均预测年龄下降至少reduce_years。

        all：
            每一个目标样本预测年龄都下降至少reduce_years。

    perturbation_tol : float
        fold change与1的差值小于该值时认为没有实际扰动。

    age_effect_tol : float
        基因单独贡献的年龄下降绝对值小于该值时过滤。

    gene_weights : pd.Series, dict or None
        基因扰动代价。None时每个基因权重相同。

    Returns
    -------
    dict
        包括基因级结果、细胞类型特征级结果、样本结果、
        扰动后的X和summary。
    """
    
    X_target = X_target.copy()
    X_target.columns = X_target.columns.astype(str)
    X_target = X_target.astype(np.float64)

    if min_fold_change <= 0:
        raise ValueError(
            "min_fold_change必须大于0。"
        )

    if max_fold_change <= 0:
        raise ValueError(
            "max_fold_change必须大于0。"
        )

    if min_fold_change >= 1:
        raise ValueError(
            "min_fold_change应小于1，才能允许基因敲低。"
        )

    if max_fold_change <= 1:
        raise ValueError(
            "max_fold_change应大于1，才能允许基因过表达。"
        )

    if constraint_mode not in {"mean", "all"}:
        raise ValueError(
            "constraint_mode必须是'mean'或'all'。"
        )

    feature_gene_map = pd.Series(
        feature_gene_map
    ).copy()

    # ========================================================
    # 1. 提取模型中可用的表达特征
    # ========================================================
    expression_features = [
        feature
        for feature in selected_expr_cols
        if feature in X_target.columns
        and feature in feature_gene_map.index
    ]

    if len(expression_features) == 0:
        raise ValueError(
            "没有找到可用于统一比例扰动的表达特征。"
        )

    feature_gene_map = feature_gene_map.loc[
        expression_features
    ].astype(str)

    # ========================================================
    # 2. 获取ElasticNet系数
    # ========================================================
    beta_all = pd.Series(
        en_model.coef_,
        index=X_target.columns,
        dtype=float
    )

    beta_expr = beta_all.loc[
        expression_features
    ]

    # 只保留模型系数非零的特征
    nonzero_mask = beta_expr.abs() > 1e-12

    expression_features = beta_expr.index[
        nonzero_mask
    ].tolist()

    beta_expr = beta_expr.loc[
        expression_features
    ]

    feature_gene_map = feature_gene_map.loc[
        expression_features
    ]

    if len(expression_features) == 0:
        raise ValueError(
            "表达特征的ElasticNet系数全部为0。"
        )

    # ========================================================
    # 3. 筛选候选基因
    # ========================================================
    available_genes = sorted(
        feature_gene_map.unique()
    )

    if candidate_genes is None:
        genes = available_genes
    else:
        candidate_genes = {
            str(gene)
            for gene in candidate_genes
        }

        genes = [
            gene
            for gene in available_genes
            if gene in candidate_genes
        ]

    if len(genes) == 0:
        raise ValueError(
            "没有可用于优化的候选基因。"
        )

    keep_feature_mask = feature_gene_map.isin(
        genes
    )

    expression_features = feature_gene_map.index[
        keep_feature_mask
    ].tolist()

    feature_gene_map = feature_gene_map.loc[
        expression_features
    ]

    beta_expr = beta_expr.loc[
        expression_features
    ]

    # ========================================================
    # 4. 获取表达特征的StandardScaler参数
    # ========================================================
    expr_mean = pd.Series(
        scaler_expr.mean_,
        index=selected_expr_cols,
        dtype=float
    )

    expr_scale = pd.Series(
        scaler_expr.scale_,
        index=selected_expr_cols,
        dtype=float
    )

    missing_scaler_features = [
        feature
        for feature in expression_features
        if feature not in expr_mean.index
    ]

    if len(missing_scaler_features) > 0:
        raise ValueError(
            "以下表达特征没有对应的scaler参数："
            f"{missing_scaler_features[:10]}"
        )

    expr_mean = expr_mean.loc[
        expression_features
    ]

    expr_scale = expr_scale.loc[
        expression_features
    ]

    if (expr_scale <= 0).any():
        raise ValueError(
            "scaler_expr.scale_中存在非正值。"
        )

    # ========================================================
    # 5. 将标准化表达还原为log1p表达
    # ========================================================
    X_expr_scaled = X_target[
        expression_features
    ].copy()

    X_expr_log1p = (
        X_expr_scaled
        * expr_scale
        + expr_mean
    )

    # 数值误差可能产生轻微负值
    X_expr_log1p = X_expr_log1p.clip(
        lower=0
    )

    # 线性表达尺度
    X_expr_linear = np.expm1(
        X_expr_log1p
    )

    # 防止非常大的log1p值导致溢出
    X_expr_linear = X_expr_linear.replace(
        [np.inf, -np.inf],
        np.nan
    )

    if X_expr_linear.isna().any().any():
        raise ValueError(
            "表达值反变换后出现NaN或无穷值。"
        )

    # ========================================================
    # 6. 基线预测年龄
    # ========================================================
    age_before = np.asarray(
        en_model.predict(X_target),
        dtype=float
    )

    n_genes = len(genes)

    gene_to_index = {
        gene: i
        for i, gene in enumerate(genes)
    }

    feature_gene_index = np.array([
        gene_to_index[
            feature_gene_map.loc[feature]
        ]
        for feature in expression_features
    ])

    # ========================================================
    # 7. 基因扰动权重
    # ========================================================
    if gene_weights is None:
        weight = np.ones(
            n_genes,
            dtype=float
        )
    else:
        gene_weights = pd.Series(
            gene_weights,
            dtype=float
        )

        weight = (
            gene_weights
            .reindex(genes)
            .fillna(1.0)
            .values
        )

    # ========================================================
    # 8. 优化变量
    #
    # log_fold_change = u_pos - u_neg
    #
    # fold_change = exp(log_fold_change)
    #
    # 使用这种分解可以将目标函数写成：
    # sum(weight * (u_pos + u_neg))
    # ========================================================
    lower_log_fc = np.log(
        min_fold_change
    )

    upper_log_fc = np.log(
        max_fold_change
    )

    max_down = abs(
        lower_log_fc
    )

    max_up = upper_log_fc

    bounds = (
        [(0.0, max_up)] * n_genes
        +
        [(0.0, max_down)] * n_genes
    )

    # ========================================================
    # 9. 根据基因fold change生成新矩阵
    # ========================================================
    def apply_gene_fold_change(
        optimization_variables
    ):
        u_pos = optimization_variables[
            :n_genes
        ]

        u_neg = optimization_variables[
            n_genes:
        ]

        log_fold_change = (
            u_pos - u_neg
        )

        fold_change = np.exp(
            log_fold_change
        )

        # 每个feature继承其所属gene的fold change
        feature_fold_change = fold_change[
            feature_gene_index
        ]

        # 在线性表达尺度乘统一比例
        X_new_linear = (
            X_expr_linear.values
            * feature_fold_change[
                np.newaxis, :
            ]
        )

        # 重新转换成log1p
        X_new_log1p = np.log1p(
            X_new_linear
        )

        # 重新标准化
        X_new_scaled = (
            X_new_log1p
            - expr_mean.values[
                np.newaxis, :
            ]
        ) / expr_scale.values[
            np.newaxis, :
        ]

        X_new = X_target.copy()

        X_new.loc[
            :,
            expression_features
        ] = X_new_scaled

        return (
            X_new,
            fold_change,
            log_fold_change
        )

    # ========================================================
    # 10. 目标函数
    #
    # 最小化总绝对log-fold-change
    # ========================================================
    def objective(
        optimization_variables
    ):
        u_pos = optimization_variables[
            :n_genes
        ]

        u_neg = optimization_variables[
            n_genes:
        ]

        return np.sum(
            weight
            * (u_pos + u_neg)
        )

    # ========================================================
    # 11. 年龄下降约束
    # ========================================================
    def age_constraint(
        optimization_variables
    ):
        X_new, _, _ = apply_gene_fold_change(
            optimization_variables
        )

        age_after = np.asarray(
            en_model.predict(X_new),
            dtype=float
        )

        age_reduction = (
            age_before - age_after
        )

        if constraint_mode == "mean":
            # scipy要求约束值 >= 0
            return (
                age_reduction.mean()
                - reduce_years
            )

        # 每个样本都至少下降reduce_years
        return (
            age_reduction
            - reduce_years
        )

    constraints = [{
        "type": "ineq",
        "fun": age_constraint
    }]

    # 初始状态：所有gene fold change = 1
    x0 = np.zeros(
        2 * n_genes,
        dtype=float
    )

    # ========================================================
    # 12. 运行非线性约束优化
    # ========================================================
    result = minimize(
        fun=objective,
        x0=x0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={
            "maxiter": maxiter,
            "ftol": ftol,
            "disp": True
        }
    )

    if not result.success:
        print("Optimization failed:")
        print(result.message)

        return {
            "success": False,
            "optimization_result": result
        }

    # ========================================================
    # 13. 提取完整优化解
    # ========================================================
    (
        X_new_full,
        fold_change_full,
        log_fold_change_full
    ) = apply_gene_fold_change(
        result.x
    )

    age_after_full = np.asarray(
        en_model.predict(X_new_full),
        dtype=float
    )

    age_reduction_full = (
        age_before - age_after_full
    )

    fold_change_series = pd.Series(
        fold_change_full,
        index=genes,
        name="fold_change"
    )

    log_fc_series = pd.Series(
        log_fold_change_full,
        index=genes,
        name="log_fold_change"
    )

    # ========================================================
    # 14. 逐基因计算年龄贡献
    #
    # 因为比例扰动在log1p空间是非线性的，
    # 这里通过一次只应用一个基因的扰动计算贡献。
    # ========================================================
    gene_age_effect = {}

    for gene in genes:
        gene_fc = fold_change_series.loc[
            gene
        ]

        if abs(gene_fc - 1.0) <= 1e-12:
            gene_age_effect[gene] = 0.0
            continue

        gene_features = feature_gene_map[
            feature_gene_map == gene
        ].index.tolist()

        X_single_gene = X_target.copy()

        old_log1p = X_expr_log1p[
            gene_features
        ]

        old_linear = np.expm1(
            old_log1p
        )

        new_linear = (
            old_linear * gene_fc
        )

        new_log1p = np.log1p(
            new_linear
        )

        new_scaled = (
            new_log1p
            - expr_mean.loc[
                gene_features
            ]
        ) / expr_scale.loc[
            gene_features
        ]

        X_single_gene.loc[
            :,
            gene_features
        ] = new_scaled

        age_single_gene = np.asarray(
            en_model.predict(
                X_single_gene
            ),
            dtype=float
        )

        # 正值表示该gene使年龄下降
        gene_age_effect[gene] = (
            age_before
            - age_single_gene
        ).mean()

    gene_age_effect = pd.Series(
        gene_age_effect
    )

    # ========================================================
    # 15. 构建基因级结果
    # ========================================================
    gene_rows = []

    for gene in genes:
        fc = fold_change_series.loc[
            gene
        ]

        log_fc = log_fc_series.loc[
            gene
        ]

        percent_change = (
            fc - 1.0
        ) * 100

        gene_features = feature_gene_map[
            feature_gene_map == gene
        ].index.tolist()

        if fc > 1:
            direction = "up"
        elif fc < 1:
            direction = "down"
        else:
            direction = "none"

        gene_rows.append({
            "gene": gene,
            "fold_change": fc,
            "log_fold_change": log_fc,
            "percent_change": percent_change,
            "abs_percent_change": abs(
                percent_change
            ),
            "direction": direction,
            "mean_age_reduction_contribution": (
                gene_age_effect.loc[gene]
            ),
            "n_celltype_features": len(
                gene_features
            ),
            "affected_features": ";".join(
                gene_features
            ),
            "weight": weight[
                gene_to_index[gene]
            ],
            "weighted_abs_log_fold_change": (
                weight[
                    gene_to_index[gene]
                ]
                * abs(log_fc)
            )
        })

    full_gene_result_df = pd.DataFrame(
        gene_rows
    ).sort_values(
        "mean_age_reduction_contribution",
        ascending=False
    )

    # ========================================================
    # 16. 过滤非常小的扰动
    # ========================================================
    gene_result_df = full_gene_result_df[
        (
            np.abs(
                full_gene_result_df[
                    "fold_change"
                ] - 1.0
            ) > perturbation_tol
        )
        &
        (
            full_gene_result_df[
                "mean_age_reduction_contribution"
            ].abs() > age_effect_tol
        )
    ].copy()

    core_genes = gene_result_df[
        "gene"
    ].tolist()

    # ========================================================
    # 17. 仅应用核心基因，重新计算年龄
    # ========================================================
    X_new_core = X_target.copy()

    feature_rows = []

    for gene in core_genes:
        fc = fold_change_series.loc[
            gene
        ]

        gene_features = feature_gene_map[
            feature_gene_map == gene
        ].index.tolist()

        for feature in gene_features:
            old_log1p = X_expr_log1p[
                feature
            ]

            old_linear = np.expm1(
                old_log1p
            )

            new_linear = (
                old_linear * fc
            )

            new_log1p = np.log1p(
                new_linear
            )

            new_scaled = (
                new_log1p
                - expr_mean.loc[feature]
            ) / expr_scale.loc[feature]

            X_new_core.loc[
                :,
                feature
            ] = new_scaled

            mean_old_linear = (
                old_linear.mean()
            )

            mean_new_linear = (
                new_linear.mean()
            )

            feature_rows.append({
                "feature": feature,
                "gene": gene,
                "beta": beta_expr.loc[
                    feature
                ],
                "fold_change": fc,
                "percent_change": (
                    fc - 1
                ) * 100,
                "direction": (
                    "up" if fc > 1
                    else "down"
                ),
                "mean_expression_before": (
                    mean_old_linear
                ),
                "mean_expression_after": (
                    mean_new_linear
                ),
                "mean_log1p_before": (
                    old_log1p.mean()
                ),
                "mean_log1p_after": (
                    new_log1p.mean()
                )
            })

    feature_result_df = pd.DataFrame(
        feature_rows
    )

    age_after_core = np.asarray(
        en_model.predict(X_new_core),
        dtype=float
    )

    age_reduction_core = (
        age_before - age_after_core
    )

    # ========================================================
    # 18. 样本级结果
    # ========================================================
    individual_result_df = pd.DataFrame({
        "sample_id": X_target.index,
        "pred_age_before": age_before,
        "pred_age_after_full": age_after_full,
        "pred_age_after_core": age_after_core,
        "age_reduction_full": age_reduction_full,
        "age_reduction_core": age_reduction_core
    })

    # ========================================================
    # 19. 结果汇总
    # ========================================================
    summary = {
        "success": True,
        "constraint_mode": constraint_mode,
        "n_samples": X_target.shape[0],
        "target_reduce_years": reduce_years,

        "mean_age_before": age_before.mean(),
        "mean_age_after_full": age_after_full.mean(),
        "mean_age_after_core": age_after_core.mean(),

        "mean_age_reduction_full": (
            age_reduction_full.mean()
        ),

        "mean_age_reduction_core": (
            age_reduction_core.mean()
        ),

        "min_age_reduction_full": (
            age_reduction_full.min()
        ),

        "min_age_reduction_core": (
            age_reduction_core.min()
        ),

        "median_age_reduction_core": (
            np.median(age_reduction_core)
        ),

        "max_age_reduction_core": (
            age_reduction_core.max()
        ),

        "n_genes_full": int(
            np.sum(
                np.abs(
                    fold_change_full - 1
                ) > perturbation_tol
            )
        ),

        "n_genes_core": (
            gene_result_df.shape[0]
        ),

        "n_celltype_features_core": (
            feature_result_df.shape[0]
        ),

        "total_abs_percent_change": (
            gene_result_df[
                "abs_percent_change"
            ].sum()
        ),

        "optimization_objective": (
            result.fun
        )
    }

    return {
        "success": True,

        "gene_perturbation_df": (
            gene_result_df
        ),

        "full_gene_result_df": (
            full_gene_result_df
        ),

        "feature_perturbation_df": (
            feature_result_df
        ),

        "individual_result_df": (
            individual_result_df
        ),

        "summary": summary,

        "X_new": X_new_core,
        "X_new_full": X_new_full,

        "optimization_result": result
    }

In [ ]:
def run_uniform_fold_change_pipeline(
    X,
    y,
    en_model,
    scaler_expr,
    selected_expr_cols,
    feature_gene_map,
    age_range=(50, 60),
    reduce_years=10,
    candidate_genes=None,
    min_fold_change=0.2,
    max_fold_change=2.0,
    constraint_mode="mean",
    perturbation_tol=0.01,
    age_effect_tol=0.01,
    save_prefix=None
):
    """
    按预测年龄筛选目标年龄段，然后运行统一比例基因扰动。
    """

    pred_age = pd.Series(en_model.predict(X), index=X.index, name="pred_age")
    y_series = pd.Series(y, index=X.index, name="true_age")

    low, high = age_range
    target_mask = ((pred_age >= low) &(pred_age < high))
    X_target = X.loc[target_mask].copy()

    if X_target.shape[0] == 0:
        raise ValueError(
            f"预测年龄{low}–{high}岁范围内没有样本。"
        )

    result = solve_uniform_fold_change_rejuvenation(
        X_target=X_target,
        en_model=en_model,
        scaler_expr=scaler_expr,
        selected_expr_cols=selected_expr_cols,
        feature_gene_map=feature_gene_map,
        reduce_years=reduce_years,
        candidate_genes=candidate_genes,
        min_fold_change=min_fold_change,
        max_fold_change=max_fold_change,
        constraint_mode=constraint_mode,
        perturbation_tol=perturbation_tol,
        age_effect_tol=age_effect_tol
    )

    if not result["success"]:
        return result

    target_info_df = pd.DataFrame({
        "sample_id": X_target.index,
        "true_age": y_series.loc[
            X_target.index
        ].values,
        "pred_age_before": pred_age.loc[
            X_target.index
        ].values
    })

    result["X_target"] = X_target
    result["target_info_df"] = target_info_df

    if save_prefix is None:
        save_prefix = (
            f"{low}-{high}_"
            f"{reduce_years}yr_"
            "uniform_fold_change"
        )

    result[
        "gene_perturbation_df"
    ].to_csv(
        f"{save_prefix}_gene_perturbation.csv",
        index=False
    )

    result[
        "feature_perturbation_df"
    ].to_csv(
        f"{save_prefix}_celltype_details.csv",
        index=False
    )

    result[
        "individual_result_df"
    ].to_csv(
        f"{save_prefix}_individual_results.csv",
        index=False
    )

    result[
        "full_gene_result_df"
    ].to_csv(
        f"{save_prefix}_full_gene_results.csv",
        index=False
    )

    return result

In [ ]:
import pandas as pd

def build_feature_gene_map(selected_features):
    """
    根据selected_features构建feature -> gene映射。

    Parameters
    ----------
    selected_features : list

        例如
        CD4Tem_S100A4_AUTS2
        CD8Tem_GZMK_NAP1L2
        MAIT_ZBTB16_ZBTB38

    Returns
    -------
    pd.Series

        index:
            feature

        value:
            gene
    """

    feature_gene_map = pd.Series(
        {
            feature: feature.rsplit("_", 1)[1]
            for feature in selected_features
        },
        name="Gene"
    )

    return feature_gene_map

selected_expr_set = set(str(col) for col in selected_expr_cols)

model_expression_features = [str(feature) for feature in selected_features if str(feature) in selected_expr_set]

feature_gene_map = build_feature_gene_map(model_expression_features)

print("模型使用的表达特征数：",len(model_expression_features))
print("模型涉及的唯一基因数：", feature_gene_map.nunique())

In [ ]:
result_50_60 = run_uniform_fold_change_pipeline(
    X=X,
    y=y,
    en_model=en_model,
    scaler_expr=scaler_expr,
    selected_expr_cols=selected_expr_cols,
    feature_gene_map=feature_gene_map,

    age_range=(50, 60),
    reduce_years=10,

    candidate_genes=None,

    # 最多敲低80%
    min_fold_change=0.2,

    # 最多过表达2倍
    max_fold_change=2.0,

    # 目标年龄段平均下降10岁
    constraint_mode="mean",

    # 相对变化小于1%时不保留
    perturbation_tol=0.01,

    age_effect_tol=0.01,

    save_prefix=(
        "50-60_10yr_"
        "uniform_fold_change"
    )
)

In [ ]:
result_40_50 = run_uniform_fold_change_pipeline(
    X=X,
    y=y,
    en_model=en_model,
    scaler_expr=scaler_expr,
    selected_expr_cols=selected_expr_cols,
    feature_gene_map=feature_gene_map,

    age_range=(40, 50),
    reduce_years=10,

    candidate_genes=None,

    # 最多敲低80%
    min_fold_change=0.2,

    # 最多过表达2倍
    max_fold_change=2.0,

    # 目标年龄段平均下降10岁
    constraint_mode="mean",

    # 相对变化小于1%时不保留
    perturbation_tol=0.01,

    age_effect_tol=0.01,

    save_prefix=(
        "40-50_10yr_"
        "uniform_fold_change"
    )
)

In [ ]:
result_60_70 = run_uniform_fold_change_pipeline(
    X=X,
    y=y,
    en_model=en_model,
    scaler_expr=scaler_expr,
    selected_expr_cols=selected_expr_cols,
    feature_gene_map=feature_gene_map,

    age_range=(60, 70),
    reduce_years=10,

    candidate_genes=None,

    # 最多敲低80%
    min_fold_change=0.2,

    # 最多过表达2倍
    max_fold_change=2.0,

    # 目标年龄段平均下降10岁
    constraint_mode="mean",

    # 相对变化小于1%时不保留
    perturbation_tol=0.01,

    age_effect_tol=0.01,

    save_prefix=(
        "60-70_10yr_"
        "uniform_fold_change"
    )
)

In [ ]:
result_70_90 = run_uniform_fold_change_pipeline(
    X=X,
    y=y,
    en_model=en_model,
    scaler_expr=scaler_expr,
    selected_expr_cols=selected_expr_cols,
    feature_gene_map=feature_gene_map,

    age_range=(70, 90),
    reduce_years=10,

    candidate_genes=None,

    # 最多敲低80%
    min_fold_change=0.2,

    # 最多过表达2倍
    max_fold_change=2.0,

    # 目标年龄段平均下降10岁
    constraint_mode="mean",

    # 相对变化小于1%时不保留
    perturbation_tol=0.01,

    age_effect_tol=0.01,

    save_prefix=(
        "70-90_10yr_"
        "uniform_fold_change"
    )
)